In [1]:
%run "./00_config.ipynb"

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE
PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None
SPARK_HOME: None
JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe
Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold
Spark version: 3.5.3
Spark master : local[*]
Parquet write test succeeded at: C:\covid_pipeline\

In [2]:
# type: ignore

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

RAW_CASES_DEATHS_DAILY = os.path.join(BRONZE_DIR, "cases_deaths.csv")

cases_schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Country/Region", StringType(), True),
    StructField("Confirmed", DoubleType(), True),
    StructField("Deaths", DoubleType(), True),
    StructField("Recovered", DoubleType(), True),
    StructField("Active", DoubleType(), True),
    StructField("New cases", DoubleType(), True),
    StructField("New deaths", DoubleType(), True),
    StructField("New recovered", DoubleType(), True),
    StructField("WHO Region", StringType(), True),
])

df_raw = (
    spark.read
    .option("header", True)
    .schema(cases_schema)
    .csv(RAW_CASES_DEATHS_DAILY)
)

print("Raw row count:", df_raw.count())
df_raw.show(5)

Raw row count: 35156
+----------+--------------+---------+------+---------+------+---------+----------+-------------+--------------------+
|      Date|Country/Region|Confirmed|Deaths|Recovered|Active|New cases|New deaths|New recovered|          WHO Region|
+----------+--------------+---------+------+---------+------+---------+----------+-------------+--------------------+
|2020-01-22|   Afghanistan|      0.0|   0.0|      0.0|   0.0|      0.0|       0.0|          0.0|Eastern Mediterra...|
|2020-01-22|       Albania|      0.0|   0.0|      0.0|   0.0|      0.0|       0.0|          0.0|              Europe|
|2020-01-22|       Algeria|      0.0|   0.0|      0.0|   0.0|      0.0|       0.0|          0.0|              Africa|
|2020-01-22|       Andorra|      0.0|   0.0|      0.0|   0.0|      0.0|       0.0|          0.0|              Europe|
|2020-01-22|        Angola|      0.0|   0.0|      0.0|   0.0|      0.0|       0.0|          0.0|              Africa|
+----------+--------------+--------

In [5]:
df_clean = (
    df_raw
    .withColumn("date", F.to_date("Date", "yyyy-MM-dd"))
    .withColumnRenamed("Country/Region", "country_name")
    .withColumnRenamed("Confirmed", "total_cases")
    .withColumnRenamed("Deaths", "total_deaths")
    .withColumnRenamed("New cases", "new_cases")
    .withColumnRenamed("New deaths", "new_deaths")
    .withColumnRenamed("WHO Region", "who_region")
    .withColumn(
        "case_fatality_rate",
        F.when(F.col("total_cases") > 0, F.col("total_deaths") / F.col("total_cases") * 100).otherwise(None)
    )
    .select("country_name", "date", "total_cases", "total_deaths", "new_cases", "new_deaths",
            "case_fatality_rate", "who_region")
)

Resolve country names to ISO-3 codes

In [9]:
# ── Manual name mapping ─────────────────────────────────────────────
name_to_iso = {
    "united states of america": "USA",
    "us": "USA",
    "republic of korea": "KOR",
    "south korea": "KOR",
    "russian federation": "RUS",
    "russia": "RUS",
    "the united kingdom": "GBR",
    "uk": "GBR",
    "iran (islamic republic of)": "IRN",
    "iran": "IRN",
    "taiwan, province of china": "TWN",
    "taiwan*": "TWN",
    "bolivia (plurinational state of)": "BOL",
    "bolivia": "BOL",
    "venezuela (bolivarian republic of)": "VEN",
    "venezuela": "VEN",
    "viet nam": "VNM",
    "vietnam": "VNM",
    "syrian arab republic": "SYR",
    "syria": "SYR",
    "democratic republic of the congo": "COD",
    "congo (kinshasa)": "COD",
    "united republic of tanzania": "TZA",
    "tanzania": "TZA",
    "republic of moldova": "MDA",
    "moldova": "MDA",
    "north macedonia": "MKD",
    "czechia": "CZE",
    "czech republic": "CZE",
    "côte d'ivoire": "CIV",
    "cote d'ivoire": "CIV",
    "cabo verde": "CPV",
    "sao tome and principe": "STP",
    "eswatini": "SWZ",
    "micronesia (federated states of)": "FSM",
    "türkiye": "TUR",
    "turkey": "TUR",
    "slovakia": "SVK",
    "puerto rico": "PRI",
    "occupied palestinian territory, including east jerusalem": "PSE",
    "west bank and gaza": "PSE",
    "egypt": "EGY",
    "réunion": "REU",
    "kosovo[1]": "XKX",
    "kosovo": "XKX",
    "martinique": "MTQ",
    "lao people's democratic republic": "LAO",
    "laos": "LAO",
    "kyrgyzstan": "KGZ",
    "guadeloupe": "GLP",
    "french guiana": "GUF",
    "jersey": "JEY",
    "curaçao": "CUW",
    "mayotte": "MYT",
    "bahamas": "BHS",
    "guernsey": "GGY",
    "saint lucia": "LCA",
    "somalia": "SOM",
    "congo": "COG",
    "congo (brazzaville)": "COG",
    "united states virgin islands": "VIR",
    "northern mariana islands (commonwealth of the)": "MNP",
    "gambia": "GMB",
    "saint martin": "MAF",
    "yemen": "YEM",
    "sint maarten": "SXM",
    "saint vincent and the grenadines": "VCT",
    "bonaire": "BES",
    "saint kitts and nevis": "KNA",
    "cook islands": "COK",
    "saint barthélemy": "BLM",
    "anguilla": "AIA",
    "saint pierre and miquelon": "SPM",
    "falkland islands (malvinas)": "FLK",
    "montserrat": "MSR",
    "sint eustatius": "BES",
    "wallis and futuna": "WLF",
    "saba": "BES",
    "niue": "NIU",
    "holy see": "VAT",
    "pitcairn islands": "PCN",
    "democratic people's republic of korea": "PRK",
    "north korea": "PRK",
    "saint helena": "SHN",
    "tokelau": "TKL",
    "burma": "MMR",
    "myanmar": "MMR",
    "cape verde": "CPV",
    "brunei": "BRN",
    "western sahara": "ESH",
    "diamond princess": None,
    "ms zaandam": None,
    # "Other" and cruise ships are not real countries — mapped to None, dropped later
}

mapping_rows = [(k, v) for k, v in name_to_iso.items()]
df_mapping = spark.createDataFrame(mapping_rows, ["cases_name_lower", "mapped_iso"])

worldbank = spark.read.parquet(SILVER_WORLDBANK)  # needed here for direct name matching

df_cases_lower = df_clean.withColumn("cases_name_lower", F.lower(F.col("country_name")))

# Try direct match against World Bank's own country names first
df_direct = df_cases_lower.join(
    worldbank.select("iso_code", F.lower(F.col("country")).alias("wb_country_lower")),
    on=[F.col("cases_name_lower") == F.col("wb_country_lower")],
    how="left"
).select(df_cases_lower["*"], F.col("iso_code").alias("iso_direct"))

# Fill gaps with manual mapping
df_with_iso = df_direct.join(df_mapping, on="cases_name_lower", how="left").withColumn(
    "iso_code",
    F.when(F.col("iso_direct").isNotNull(), F.col("iso_direct")).otherwise(F.col("mapped_iso"))
)

unmatched = df_with_iso.filter(F.col("iso_code").isNull())
print(f"Still unmatched: {unmatched.count()}")
unmatched.select("country_name").distinct().show(50, truncate=False)

Still unmatched: 0
+------------+
|country_name|
+------------+
+------------+



In [10]:
# Drop rows that genuinely aren't countries (cruise ships etc.), drop helper columns, finalize
df_final = (
    df_with_iso
    .filter(F.col("iso_code").isNotNull())
    .drop("iso_direct", "cases_name_lower", "mapped_iso", "country_name")
)

print("Final row count:", df_final.count())
df_final.show(5)

Final row count: 35156
+----------+-----------+------------+---------+----------+------------------+--------------------+--------+
|      date|total_cases|total_deaths|new_cases|new_deaths|case_fatality_rate|          who_region|iso_code|
+----------+-----------+------------+---------+----------+------------------+--------------------+--------+
|2020-01-22|        0.0|         0.0|      0.0|       0.0|              NULL|              Europe|     ARM|
|2020-01-22|        0.0|         0.0|      0.0|       0.0|              NULL|              Europe|     AUT|
|2020-01-22|        0.0|         0.0|      0.0|       0.0|              NULL|              Europe|     AZE|
|2020-01-22|        0.0|         0.0|      0.0|       0.0|              NULL|Eastern Mediterra...|     BHR|
|2020-01-22|        0.0|         0.0|      0.0|       0.0|              NULL|            Americas|     BRA|
+----------+-----------+------------+---------+----------+------------------+--------------------+--------+
only 

In [11]:
df_final.write.mode("overwrite").parquet(SILVER_CASES_DEATHS)
print("Written to:", SILVER_CASES_DEATHS)

Written to: C:\covid_pipeline\silver\cases_deaths
